# SpeakAI-Eval — Inference (Run Only)

Notebook này chạy toàn bộ hệ thống đánh giá phát âm và hội thoại IELTS Speaking AI trên Kaggle (GPU NVIDIA T4 x 2 hoặc P100):
- **ASR Engine**: `Whisper Large-v3-Turbo` (`openai/whisper-large-v3-turbo`, Beam Search 5, Dynamic Context Prompting + Word Timestamps)
- **Speaker Diarization**: `TwoSpeakerSplitter` (ERes2Net-Large In-Domain Clustering + Dual Reference Role Mapping + DeepFilterNet3)
- **Pronunciation Scoring**: `SpeechOcean762` (WavLM-Large + Phoneme Graph Attention GAT)
- **Phonological Error Diagnosis**: `L2-MDD` (WavLM-Large + CTC Forced Alignment theo chuẩn ASHA)
- **ASR Error Correction & Linguistic Scoring**: `deepseek-v4.1-flash` + `spaCy (en_core_web_sm)` + `LanguageTool` + `wordfreq`
- **Classroom Language Timeline**: `DeepFilterNet3 Denoise -> RmsVad Speech Gate -> Whisper Turbo Batched 1-Step Acoustic LID (EN / VI / Silence, Zero Hallucination)`

Mã nguồn, models và checkpoint được đọc trực tiếp từ Kaggle Dataset hoặc tải tự động.


In [ ]:
# Cài đặt thư viện
import subprocess, sys, os
try:
    import numpy
    np_ver = numpy.__version__
except: np_ver = '2.0.2'
print('Installing Rust (required for DeepFilterNet on Python 3.12)...')
os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
pkgs = [
    'torch>=2.1.0', 'torchaudio>=2.1.0', 'torch-geometric>=2.4.0',
    'peft>=0.7.0', 'pyyaml>=6.0.1',
    'numpy>=1.24.0', 'scipy>=1.10.0', 'soundfile>=0.12.1', 'nltk>=3.8.1',
    'python-dotenv>=1.0.0', 'deepfilternet', 'noisereduce>=3.0.0',
    'addict', 'modelscope', 'speechbrain>=1.0.0', 'huggingface_hub>=0.23.0',
    'hf-transfer', 'safetensors>=0.4.0', 'fastapi', 'uvicorn',
    'python-multipart', 'requests>=2.31.0', 'nest-asyncio',
    'language-tool-python', 'spacy', 'wordfreq',
]
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', 'transformers>=4.45.0', 'accelerate>=0.26.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', f'numpy=={np_ver}'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=False)
# Tải spaCy en_core_web_sm cho bộ chấm GRA/LR
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=False)
import transformers
print(f'Libraries installed! Transformers version: {transformers.__version__}')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('Downloading cloudflared...')
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared', shell=True, check=True)
    subprocess.run('chmod +x /usr/local/bin/cloudflared', shell=True, check=True)


---
## Khởi tạo Pipeline từ Kaggle Input

In [ ]:
import sys, os, types
os.environ['LLM_API_KEY'] = 'sk_g4V9GbG576mtTdxGNgZ9K7lczVY13x7B'
try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception: pass
# Ensure torchaudio.backend compatibility for DeepFilterNet on torchaudio >= 2.1
import torchaudio
if not hasattr(torchaudio, 'backend'):
    torchaudio.backend = types.ModuleType('torchaudio.backend')
    sys.modules['torchaudio.backend'] = torchaudio.backend
if not hasattr(torchaudio.backend, 'common'):
    torchaudio.backend.common = types.ModuleType('torchaudio.backend.common')
    sys.modules['torchaudio.backend.common'] = torchaudio.backend.common
    torchaudio.backend.common.AudioMetaData = getattr(torchaudio, 'AudioMetaData', type('AudioMetaData', (), {}))
SOURCE_DIR = None
MODEL_DIR = None
WHISPER_PATH = None
PRONUNCIATION_PATH = None
PRONUNCIATION_FALLBACK_PATH = None
L2_MDD_PATH = None
for search_dir in ['/kaggle/working', '/kaggle/input']:
    for root, dirs, files in os.walk(search_dir):
        if 'pretrained_models' in dirs and 'wavlm-large' in os.listdir(os.path.join(root, 'pretrained_models')):
            if MODEL_DIR is None: MODEL_DIR = root
        if 'infer' in dirs and 'speaker-diarize' in dirs:
            if SOURCE_DIR is None: SOURCE_DIR = root
        if 'whisper-large-v3-turbo' in dirs:
            w_cand = os.path.join(root, 'whisper-large-v3-turbo')
            if WHISPER_PATH is None and os.path.isdir(w_cand) and os.path.isfile(os.path.join(w_cand, 'config.json')):
                WHISPER_PATH = w_cand
        if 'pronunciation.pt' in files:
            if PRONUNCIATION_PATH is None: PRONUNCIATION_PATH = os.path.join(root, 'pronunciation.pt')
        if 'best_model.pt' in files and 'l2_mdd' not in root.lower():
            if PRONUNCIATION_FALLBACK_PATH is None: PRONUNCIATION_FALLBACK_PATH = os.path.join(root, 'best_model.pt')
        if 'l2_mdd_best.pt' in files:
            if L2_MDD_PATH is None: L2_MDD_PATH = os.path.join(root, 'l2_mdd_best.pt')
        elif 'l2_mdd.pt' in files:
            if L2_MDD_PATH is None: L2_MDD_PATH = os.path.join(root, 'l2_mdd.pt')
if not PRONUNCIATION_PATH and PRONUNCIATION_FALLBACK_PATH:
    PRONUNCIATION_PATH = PRONUNCIATION_FALLBACK_PATH
if not MODEL_DIR:
    raise FileNotFoundError("Không tìm thấy thư mục 'pretrained_models/wavlm-large' trong dataset model.")
if not SOURCE_DIR:
    raise FileNotFoundError("Không tìm thấy source code (infer, speaker-diarize) trong dataset setup.")
if not PRONUNCIATION_PATH:
    raise FileNotFoundError("Không tìm thấy file 'pronunciation.pt' (hoặc 'best_model.pt') trong bất kỳ dataset nào.")
if L2_MDD_PATH:
    print(f'✅ Tìm thấy L2-MDD checkpoint: {L2_MDD_PATH}')
else:
    print('⚠️ Không tìm thấy l2_mdd_best.pt — chỉ dùng pronunciation model')
if WHISPER_PATH:
    print(f'✅ Tìm thấy Whisper Large-v3-Turbo offline tại: {WHISPER_PATH}')
else:
    print('ℹ️ Không thấy Whisper Large-v3-Turbo offline — sẽ tự tải openai/whisper-large-v3-turbo (~1.6GB) qua hf-transfer.')
sys.path.insert(0, SOURCE_DIR)
sys.path.insert(0, f'{SOURCE_DIR}/speaker-diarize')
os.chdir(SOURCE_DIR)

# Lưu cache HuggingFace vào /kaggle/working (ổn định & có quyền ghi)
os.environ['HF_HOME'] = '/kaggle/working/hf_cache'

import torch
import yaml
from infer.pipeline import SpeakingPipeline
from speaker_diarize.embedding import ERes2NetEmbedder

num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if num_gpus >= 2:
    asr_dev, score_dev = 'cuda:0', 'cuda:1'
elif num_gpus == 1:
    asr_dev, score_dev = 'cuda:0', 'cuda:0'
else:
    asr_dev, score_dev = 'cpu', 'cpu'

print(f'Writing runtime config (GPUs detected: {num_gpus}, asr={asr_dev}, score={score_dev})...')
with open(f'{SOURCE_DIR}/configs/pronunciation.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
whisper_name = WHISPER_PATH if WHISPER_PATH else cfg.get('asr', {}).get('model_name', 'openai/whisper-large-v3-turbo')
cfg['asr']['engine'] = 'whisper'
cfg['asr']['model_name'] = whisper_name
cfg['asr']['device'] = asr_dev
cfg['inference']['device'] = score_dev
cfg['project']['device'] = asr_dev
cfg['wavlm']['model_name'] = f'{MODEL_DIR}/pretrained_models/wavlm-large'
cfg['paths']['pronunciation_checkpoint'] = PRONUNCIATION_PATH
if L2_MDD_PATH:
    cfg['paths']['l2_mdd_checkpoint'] = L2_MDD_PATH
os.makedirs('/kaggle/working/tmp_configs', exist_ok=True)
tmp_config = '/kaggle/working/tmp_configs/pronunciation.yaml'
with open(tmp_config, 'w', encoding='utf-8') as f:
    yaml.dump(cfg, f)

if num_gpus >= 2:
    print('=== KHỞI TẠO PIPELINE PARALLELISM TRÊN 2 GPU (NVIDIA T4 x 2) ===')
    print(f'  GPU 0 ({asr_dev}): Whisper Large-v3-Turbo + TwoSpeakerSplitter (ERes2Net + DFN3) + Registration Embedder')
    print(f'  GPU 1 ({score_dev}): SpeechOcean762 (WavLM-large + GAT) + L2-MDD (WavLM-large + CTC)')
elif num_gpus == 1:
    print(f'=== KHỞI TẠO PIPELINE TRÊN 1 GPU ({asr_dev}) ===')
else:
    print('⚠️ Không phát hiện GPU CUDA — đang chạy trên CPU (rất chậm).')

print(f'\n[1/3] Loading Registration Model (ERes2Net) on {asr_dev}...')
extract_embedder = ERes2NetEmbedder(device=asr_dev)
print(f'✅ Registration Embedder ready on {asr_dev}!')

print(f'\n[2/3] Loading SpeakingPipeline (Whisper & Diarize on {asr_dev}, Scoring on {score_dev})...')
pipeline = SpeakingPipeline(
    config_path=tmp_config,
    device=score_dev,
    asr_device=asr_dev,
    diarize_device=asr_dev,
    l2_mdd_ckpt=L2_MDD_PATH,
)
print('✅ SpeakingPipeline ready!')

print('\n[3/3] Preloading spaCy (en_core_web_sm) & LanguageTool JVM (tránh chờ 30-45s ở Task đầu tiên)...')
from infer.hybrid_nlp_scorer import get_spacy_nlp, get_language_tool
_nlp_ok = get_spacy_nlp() is not None
_lt_ok = get_language_tool() is not None
print(f'✅ NLP Preload done -> spaCy={_nlp_ok} | LanguageTool={_lt_ok}')

print('🎉 ALL SYSTEMS READY!')


---
## Chạy Thử Đánh Giá Trực Tiếp (Interactive Evaluation)
Bạn có thể chạy thử pipeline trực tiếp trên file audio trong Kaggle trước khi khởi động API server:


In [ ]:
# Ví dụ đánh giá trực tiếp file hội thoại mẫu Du.mp3 hoặc Sophia.mp3
sample_audio = f'{SOURCE_DIR}/sample_audio/data_AI_Speaking/audio_2_nguoi/Sophia.mp3'
if not os.path.exists(sample_audio):
    sample_audio = f'{SOURCE_DIR}/docs/data_AI_Speaking/audio_2_nguoi/Sophia.mp3'
if os.path.exists(sample_audio):
    print(f'Đang đánh giá file mẫu: {sample_audio}...')
    eval_res = pipeline.assess_conversation(sample_audio, use_asr=True, score_teacher=False)
    student = eval_res.get('student') or eval_res.get('speakers', {}).get('B', {})
    sents = student.get('sentences', [])
    scored_turns = [s for s in sents if s.get('scored')]
    print(f'\n✅ Tổng số lượt nói được chấm: {len(scored_turns)} / {len(sents)} turns')
    print('--- Bảng điểm 4 tiêu chí IELTS Speaking (Band 1.0 - 9.0) ---')
    for k in ('total', 'p', 'fc', 'lr', 'gra'):
        if k in student.get('scores', {}):
            print(f'  {k.upper()}: {student["scores"][k]}')
else:
    print('Bạn có thể truyền đường dẫn file audio bất kỳ vào pipeline.assess_conversation(audio_path)')


---
## Khởi chạy Backend API (FastAPI + Cloudflare Tunnel)
- Khởi tạo Cloudflare Tunnel công khai kết nối máy chủ AI.
- **Không cần cấu hình key Supabase trên Kaggle**: Sao chép link Cloudflare in ra ở bên dưới và dán vào: Trang Quản Trị (Admin) > Cài Đặt Hệ Thống > Kết Nối GPU.


In [ ]:
# Khởi chạy Cloudflare Tunnel & FastAPI Server (Không cần key Supabase)
import os
os.environ['LLM_API_KEY'] = 'sk_g4V9GbG576mtTdxGNgZ9K7lczVY13x7B'

import env_config
import backend_api
import builtins

_llm_key = env_config.get_llm_api_key()
try:
    import spacy
    spacy.load('en_core_web_sm')
    _spacy_ok = True
except Exception:
    _spacy_ok = False
print(f'Preflight -> LLM_API_KEY ({env_config.DEFAULT_LLM_MODEL}): {bool(_llm_key)} | spaCy en_core_web_sm: {_spacy_ok}')

# Start Cloudflare Tunnel
PUBLIC_URL = backend_api.start_cloudflare_tunnel(8000)
backend_api.PUBLIC_URL = PUBLIC_URL
if PUBLIC_URL:
    print('\n' + '=' * 80)
    print(f'🚀 CLOUDFLARE PUBLIC API URL: {PUBLIC_URL}')
    print('👉 Hãy sao chép URL trên và dán vào: Web SpeakAI > Quản trị (Admin) > Cài đặt hệ thống > Kết nối GPU')
    print('=' * 80 + '\n')
else:
    print('⚠️ Running without Cloudflare tunnel (localhost only)')

# Register notebook globals so backend_api can find them
builtins.pipeline = pipeline
builtins.extract_embedder = extract_embedder

# Start FastAPI server in background thread
import threading, uvicorn
config = uvicorn.Config(app=backend_api.app, host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
print('✅ FastAPI Server is running!')
import time
try:
    while server_thread.is_alive():
        time.sleep(1)
except KeyboardInterrupt:
    print('Shutting down FastAPI Server...')
    server.should_exit = True
    server_thread.join(timeout=5)
